# Build the L2 latency bundle (run once, on Colab/Kaggle with internet)

Builds the model artifacts that `latency_bench_kaggle.ipynb` will consume offline. After this notebook finishes, download the produced files and assemble them with the locally-built parapet-runner wheel and the locally-built corpus into a `parapet-l2-bench/` directory, then upload as a Kaggle dataset.

**Outputs** (all under `mdeberta-onnx-export/`):
* `model.onnx` — fp32 ONNX export of `microsoft/mdeberta-v3-base`.
* `model.int8.onnx` — int8 dynamic-quantized.
* `model.int8.sha256` — digest of the int8 artifact.
* `revision.txt` — immutable HF commit SHA for the model snapshot used.
* tokenizer files (`tokenizer.json`, `tokenizer_config.json`, `spm.model`, `special_tokens_map.json`) written by `optimum-cli` during export.

**Final dataset structure** (assemble locally, upload as one Kaggle dataset):
```
parapet-l2-bench/
  parapet_runner-0.1.0-py3-none-any.whl    (built locally with python -m build)
  model/                                   (everything from mdeberta-onnx-export/)
  corpus/
    l2_latency_v8_train_val_stratified.jsonl
    l2_latency_v8_train_val_stratified.jsonl.manifest.json
```

## 1. Install build deps

`optimum[onnx]` (not the deprecated `optimum[exporters]` extra). `huggingface_hub` for resolving the immutable revision SHA. `onnxruntime` is in Kaggle's base image but pinned here in case Colab's image is leaner.

In [ ]:
!pip install -q "optimum[onnx]>=1.20" "huggingface_hub>=0.25" "onnxruntime>=1.17"

## 2. Export mDeBERTa to ONNX

`optimum-cli export onnx --model <repo> --task feature-extraction --opset 17 <output_dir>` — current Optimum CLI shape per the HF Optimum-ONNX docs.

In [ ]:
!optimum-cli export onnx \
    --model microsoft/mdeberta-v3-base \
    --task feature-extraction \
    --opset 17 \
    mdeberta-onnx-export

In [ ]:
# Sanity: confirm model.onnx and the tokenizer files actually landed.
!ls -la mdeberta-onnx-export/

## 3. Quantize to int8 (dynamic)

`direction.md` Phase 0.6 measures int8 specifically. Dynamic quantization needs no calibration dataset.

In [ ]:
from onnxruntime.quantization import quantize_dynamic, QuantType
quantize_dynamic(
    "mdeberta-onnx-export/model.onnx",
    "mdeberta-onnx-export/model.int8.onnx",
    weight_type=QuantType.QInt8,
)
print("quantized OK")

## 4. Hash the int8 artifact

The bench refuses to load if the on-disk hash doesn't match the configured one — this digest goes into `--onnx-sha256` downstream.

In [ ]:
import hashlib
h = hashlib.sha256()
with open("mdeberta-onnx-export/model.int8.onnx", "rb") as f:
    while chunk := f.read(65536):
        h.update(chunk)
ONNX_SHA = h.hexdigest()
open("mdeberta-onnx-export/model.int8.sha256", "w").write(ONNX_SHA)
print(f"int8 model SHA-256: {ONNX_SHA}")

## 5. Resolve immutable HF revision SHA

Pin to the actual commit, not `'main'` or a tag. The revision lands in `revision.txt` so the bench can read it without an HF Hub call at runtime.

In [ ]:
from huggingface_hub import HfApi
HF_REV = HfApi().model_info("microsoft/mdeberta-v3-base").sha
open("mdeberta-onnx-export/revision.txt", "w").write(HF_REV)
print(f"mDeBERTa-v3-base revision SHA: {HF_REV}")

## 6. Fail-fast validation

Before declaring the bundle ready, exercise every artifact end-to-end: file existence, tokenizer load (local-only, no HF Hub fetch), encode a sample, ORT inference on the int8 model using the model's declared input names, and a shape check on `revision.txt`. If any of this breaks, the bundle is broken and we want to surface it here, not on Kaggle.

In [ ]:
import os
import re
import numpy as np
import onnxruntime as ort
from transformers import AutoTokenizer

for required in [
    "mdeberta-onnx-export/model.onnx",
    "mdeberta-onnx-export/model.int8.onnx",
    "mdeberta-onnx-export/model.int8.sha256",
    "mdeberta-onnx-export/revision.txt",
]:
    assert os.path.isfile(required), f"missing artifact: {required}"

# Tokenizer must load from the local export dir without touching HF Hub.
tok = AutoTokenizer.from_pretrained(
    "mdeberta-onnx-export", local_files_only=True, use_fast=True
)
encoded = tok(
    "hello world",
    return_tensors="np",
    padding=False,
    truncation=True,
    max_length=128,
)
assert "input_ids" in encoded and encoded["input_ids"].shape[1] > 0

# ORT must load the int8 model on CPU and accept the tokenizer's outputs
# under the ONNX-declared input names.
sess = ort.InferenceSession(
    "mdeberta-onnx-export/model.int8.onnx",
    providers=["CPUExecutionProvider"],
)
input_names = {inp.name for inp in sess.get_inputs()}
feed = {name: np.asarray(encoded[name]) for name in input_names if name in encoded}
missing = input_names - feed.keys()
assert not missing, f"tokenizer missing required ONNX inputs: {missing}"
output = sess.run(None, feed)
assert output and output[0].size > 0, "ORT produced empty output"

# revision.txt must be a 40-char git commit SHA.
rev = open("mdeberta-onnx-export/revision.txt").read().strip()
assert re.fullmatch(r"[0-9a-fA-F]{40}", rev), f"revision.txt not a 40-char hex SHA: {rev!r}"

print("validation OK")
print(f"  ONNX inputs: {sorted(input_names)}")
print(f"  out shape:   {output[0].shape}")
print(f"  revision:    {rev}")

## 6. Final artifact listing

Download every file under `mdeberta-onnx-export/` and copy them into the `model/` subdirectory of your local `parapet-l2-bench/` bundle directory. Then add the wheel and the corpus JSONL (+ its manifest) per the structure in cell 0.

In [ ]:
!ls -la mdeberta-onnx-export/
print()
print("Bundle assembly checklist (do this locally before uploading to Kaggle):")
print("  parapet-l2-bench/")
print("    parapet_runner-0.1.0-py3-none-any.whl")
print("    model/        <- copy everything from mdeberta-onnx-export/")
print("    corpus/")
print("      l2_latency_v8_train_val_stratified.jsonl")
print("      l2_latency_v8_train_val_stratified.jsonl.manifest.json")